# Fraud Detection System
## 01 — Business Understanding

| | |
|---|---|
| **Author** | Jose Lopez Pino |
| **Framework** | CRISP-DM + Lean |
| **Phase** | 1 — Business Understanding |
| **Project** | fraud-detection · applied-data-science-portfolio |
| **Dataset** | Credit Card Fraud Detection (ULB, Kaggle) |
| **Date** | 2026-04 |

---

> **Executive Summary:**
> This notebook defines the business problem, the decision to support, and the cost framework
> that governs all modeling choices in this project. The core insight is that fraud detection
> is not an accuracy problem — it is an asymmetric cost problem where false negatives
> (missed fraud) are far more expensive than false positives (blocked legitimate transactions).
> The output is a Problem Statement Canvas and an Expected Loss Framework that will drive
> threshold calibration in notebook 05.

## Table of Contents

1. [CRISP-DM Phase 1 — Business Understanding](#1-crisp-dm-phase-1--business-understanding)
2. [Problem Statement Canvas](#2-problem-statement-canvas)
3. [The Cost Asymmetry Framework](#3-the-cost-asymmetry-framework)
4. [Expected Loss Framework](#4-expected-loss-framework)
5. [Success Metrics Definition](#5-success-metrics-definition)
6. [Modeling Roadmap](#6-modeling-roadmap)
7. [LEAN Filter — Waste Elimination Review](#7-lean-filter--waste-elimination-review)
8. [Decisions Log](#8-decisions-log)
9. [Next Steps — Notebook 02 Preview](#9-next-steps--notebook-02-preview)

---
## 1. CRISP-DM Phase 1 — Business Understanding

### 1.1 Why this project exists

Financial institutions lose billions annually to fraud — but the problem is not just
undetected fraud. It is the **double cost of errors**:

- **False Negatives (FN):** Fraud that passes through undetected → direct financial loss
- **False Positives (FP):** Legitimate transactions blocked → customer friction, churn risk, operational cost

A model optimized for accuracy will almost always fail here, because with a fraud rate
of ~0.17%, a model that predicts "legitimate" for every transaction achieves 99.83% accuracy
— and catches zero fraud.

### 1.2 The real business question

The question is not *"can we detect fraud?"* — it is:

> **At what classification threshold does the model minimize total expected loss,
> given the asymmetric costs of FN and FP?**

This is a decision optimization problem disguised as a classification problem.
The Industrial Engineering framing applies directly: minimize total cost under constraints.

### 1.3 Stakeholders

| Stakeholder | What they need | Primary metric |
|---|---|---|
| **Risk / Fraud team** | Catch as many fraudulent transactions as possible | Recall (fraud class) |
| **Customer experience team** | Minimize false blocks on legitimate customers | Precision (fraud class) |
| **CFO / Finance** | Minimize total expected monetary loss | Expected Loss (EL) |
| **ML team** | A model that generalizes beyond the training window | AUC-PR |

The CFO's metric is the integrating objective — it forces a quantified tradeoff between
recall and precision that the other stakeholders cannot resolve alone.

---
## 2. Problem Statement Canvas

| # | Element | Content |
|---|---|---|
| 1 | **Business Problem** | Financial institutions face two simultaneous failure modes: fraud passing undetected (direct loss) and legitimate transactions blocked (customer friction + churn). Standard accuracy metrics hide both failures. |
| 2 | **Business Impact** | Global card fraud losses exceed USD 30B annually. A naive model (predict all legitimate) achieves 99.83% accuracy and catches zero fraud — exposing the institution to full FN cost. |
| 3 | **Decision to Support** | At what classification threshold should a transaction be flagged as fraudulent, given a defined cost ratio between FN and FP? |
| 4 | **Analytical Question** | Can a supervised classification model, with proper handling of class imbalance and threshold calibrated by expected cost, achieve Recall ≥ 0.80 and Precision ≥ 0.70 on the fraud class? |
| 5 | **Success Metrics** | Recall ≥ 0.80 (fraud class) · Precision ≥ 0.70 (fraud class) · AUC-PR ≥ 0.85 · Expected Loss < baseline (naive classifier) · Threshold defined by cost function, not default 0.5 |
| 6 | **Proposed Approach** | Supervised classification: Logistic Regression (baseline) → Random Forest → XGBoost. Imbalance handling: class_weight vs SMOTE comparison. Threshold calibration via cost function. Isolation Forest as unsupervised benchmark. |

---
## 3. The Cost Asymmetry Framework

### 3.1 Why default threshold (0.5) is wrong

The default classification threshold of 0.5 assumes symmetric costs:
misclassifying a fraud is equally bad as misclassifying a legitimate transaction.

In fraud detection, this assumption is always wrong.

### 3.2 Cost matrix

| Prediction \ Reality | Legitimate (0) | Fraud (1) |
|---|---|---|
| **Predicted Legitimate (0)** | ✅ True Negative — no cost | ❌ False Negative — `cost_fn` |
| **Predicted Fraud (1)** | ⚠️ False Positive — `cost_fp` | ✅ True Positive — no cost |

### 3.3 Cost definitions

```
cost_fn = average fraudulent transaction amount
          + investigation cost (if any)
          → typically USD 100–500 per transaction in retail banking

cost_fp = customer service intervention cost
          + estimated churn probability × customer lifetime value
          → typically USD 5–20 per blocked transaction
```

### 3.4 Cost ratio

$$\text{cost\_ratio} = \frac{\text{cost\_fn}}{\text{cost\_fp}}$$

A cost ratio of 10 means missing one fraud is 10x more costly than blocking one
legitimate transaction. The threshold that minimizes Expected Loss will shift
toward higher recall (lower threshold) as this ratio increases.

In notebook 05, this ratio becomes an interactive parameter — the Streamlit app
will expose it as a slider so business stakeholders can explore the tradeoff
without touching code.

---
## 4. Expected Loss Framework

### 4.1 Definition

The Expected Loss (EL) of a classifier at a given threshold $t$ is:

$$EL(t) = FN(t) \times \text{cost\_fn} + FP(t) \times \text{cost\_fp}$$

Where:
- $FN(t)$ = number of fraud transactions classified as legitimate at threshold $t$
- $FP(t)$ = number of legitimate transactions classified as fraud at threshold $t$

### 4.2 Optimal threshold

$$t^* = \arg\min_t \ EL(t)$$

This is the threshold that will be used for final model evaluation in notebook 05.
It replaces the arbitrary default of 0.5.

### 4.3 Comparison baselines

The model must beat three baselines to demonstrate real value:

| Baseline | Description | Expected Loss |
|---|---|---|
| **Naive — all legitimate** | Predict 0 for every transaction | EL = N_fraud × cost_fn |
| **Naive — all fraud** | Predict 1 for every transaction | EL = N_legit × cost_fp |
| **Random classifier** | Random prediction at fraud rate | EL = interpolation of above |
| **Our model at t*** | Optimized threshold | EL < all baselines |

A model that does not beat the naive baseline has no business value — regardless of its AUC.

---
## 5. Success Metrics Definition

### 5.1 Primary metrics (fraud class)

| Metric | Target | Why this metric |
|---|---|---|
| **Recall** | ≥ 0.80 | Fraction of actual fraud caught — the core business requirement |
| **Precision** | ≥ 0.70 | Fraction of fraud flags that are real — controls false positive cost |
| **AUC-PR** | ≥ 0.85 | Area under Precision-Recall curve — primary ranking metric for imbalanced data |
| **Expected Loss** | < naive baseline | Total monetary cost at optimal threshold — integrating business metric |

### 5.2 Why AUC-PR, not AUC-ROC

With a fraud rate of ~0.17%, AUC-ROC is optimistic and misleading:
- The True Negative Rate (specificity) is always high because there are so many negatives
- AUC-ROC can be 0.97 while the model is nearly useless at catching fraud

AUC-PR focuses on the minority class (fraud) and is insensitive to the large
number of true negatives. It is the correct metric for this problem.

### 5.3 What we explicitly do NOT optimize for

| Metric | Why we ignore it |
|---|---|
| **Accuracy** | Misleading with 0.17% fraud rate — naive classifier scores 99.83% |
| **AUC-ROC as primary** | Optimistic under extreme imbalance |
| **F1 at threshold 0.5** | Default threshold ignores cost asymmetry |

---
## 6. Modeling Roadmap

### 6.1 CRISP-DM arc for this project

| Notebook | Phase | Key Output |
|---|---|---|
| **01 — Business Understanding** | Define problem + cost framework | Problem Statement Canvas · Expected Loss Framework |
| **02 — Data Understanding** | Explore dataset structure + fraud patterns | EDA report · data quality assessment |
| **03 — Data Preparation** | Clean + transform + imbalance strategy | Processed dataset · imbalance comparison |
| **04 — Modeling** | Train LR → RF → XGBoost + Isolation Forest | Trained models · AUC-PR comparison |
| **05 — Evaluation** | Threshold calibration + Expected Loss | Optimal threshold · EL vs baselines |
| **06 — Deployment** | Model card + export + Streamlit app | `model_v1_2026-04.pkl` · app with cost slider |

### 6.2 Model selection rationale

| Model | Role | Rationale |
|---|---|---|
| **Logistic Regression** | Baseline | Interpretable, fast, establishes minimum bar |
| **Random Forest** | Strong baseline | Robust to imbalance with class_weight, handles non-linearity |
| **XGBoost** | Primary candidate | Industry standard for tabular fraud detection |
| **Isolation Forest** | Unsupervised benchmark | Tests whether labeled data is necessary — shows paradigm judgment |

### 6.3 Imbalance strategies to compare

| Strategy | Description | Will test? |
|---|---|---|
| No balancing | Raw class distribution | ✅ |
| `class_weight='balanced'` | Sklearn built-in weighting | ✅ |
| SMOTE | Synthetic minority oversampling | ✅ |
| ADASYN | Adaptive synthetic sampling | ✅ optional |

In [1]:
# ===== Environment Setup =====
import warnings
warnings.formatwarning = lambda msg, *args, **kwargs: f'Warning: {msg}\n'
warnings.simplefilter('always')

# === Standard library ===
from pathlib import Path

# === Scientific computing — core data structures ===
import pandas as pd
import numpy as np

# === Visualization — base plotting ===
import matplotlib.pyplot as plt

# === Visualization — statistical plotting ===
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Blues_d')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_RAW        = Path('../data/raw')
DATA_PROCESSED  = Path('../data/processed')
DATA_FINAL      = Path('../data/final')
REPORTS_FIGURES = Path('../reports/figures')
REPORTS_FIGURES.mkdir(parents=True, exist_ok=True)

print('Environment ready.')
print(f'Data path   : {DATA_RAW}')
print(f'Figures path: {REPORTS_FIGURES}')

Environment ready.
Data path   : ..\data\raw
Figures path: ..\reports\figures


In [2]:
# ===== Cost Framework — parametric definition =====
# Business assumption: cost of missing a fraud (FN) vs blocking a legit transaction (FP)
# These values will be updated in notebook 05 based on actual dataset transaction amounts

COST_FN = 300   # USD — estimated average fraudulent transaction amount
COST_FP = 15    # USD — estimated cost of blocking a legitimate transaction

COST_RATIO = COST_FN / COST_FP

print('=== Cost Asymmetry Framework ===')
print(f'Cost of missing fraud (FN)         : USD {COST_FN}')
print(f'Cost of blocking legitimate (FP)   : USD {COST_FP}')
print(f'Cost ratio (FN / FP)               : {COST_RATIO:.1f}x')
print()
print('Interpretation:')
print(f'  Missing 1 fraud = equivalent to blocking {COST_RATIO:.0f} legitimate transactions')
print('  → Optimal threshold will shift toward higher recall (lower than 0.5)')

=== Cost Asymmetry Framework ===
Cost of missing fraud (FN)         : USD 300
Cost of blocking legitimate (FP)   : USD 15
Cost ratio (FN / FP)               : 20.0x

Interpretation:
  Missing 1 fraud = equivalent to blocking 20 legitimate transactions
  → Optimal threshold will shift toward higher recall (lower than 0.5)


In [3]:
# ===== Expected Loss — baseline projections =====
# Estimated from known dataset statistics (284,807 transactions, 492 fraud)
# Will be recalculated with real data in notebook 05

N_TOTAL = 284_807
N_FRAUD = 492
N_LEGIT = N_TOTAL - N_FRAUD
FRAUD_RATE = N_FRAUD / N_TOTAL

# Baseline 1: predict all legitimate (catches zero fraud)
el_naive_all_legit = N_FRAUD * COST_FN

# Baseline 2: predict all fraud (blocks every transaction)
el_naive_all_fraud = N_LEGIT * COST_FP

print('=== Expected Loss Baselines ===')
print(f'Dataset: {N_TOTAL:,} transactions | {N_FRAUD:,} fraud ({FRAUD_RATE:.4%})')
print()
print(f'Baseline — predict ALL legitimate : USD {el_naive_all_legit:>12,.0f}')
print(f'Baseline — predict ALL fraud      : USD {el_naive_all_fraud:>12,.0f}')
print()
print('Our model must beat the lower of these two baselines.')
print(f'Target: EL < USD {min(el_naive_all_legit, el_naive_all_fraud):,.0f}')

=== Expected Loss Baselines ===
Dataset: 284,807 transactions | 492 fraud (0.1727%)

Baseline — predict ALL legitimate : USD      147,600
Baseline — predict ALL fraud      : USD    4,264,725

Our model must beat the lower of these two baselines.
Target: EL < USD 147,600


---
## 7. LEAN Filter — Waste Elimination Review

| LEAN Question | Answer | Action |
|---|---|---|
| Does every analysis link to a business decision? | ✅ Yes — cost framework directly drives threshold choice in nb05 | Proceed |
| Is there redundancy between sections? | ✅ No — each section adds a distinct layer: problem → cost → metrics → roadmap | Proceed |
| Is n sufficient for planned tests? | ✅ Yes — 284,807 transactions, cross-validation planned | Proceed |
| Is there a simpler way to answer the question? | ✅ No — cost asymmetry requires explicit framework; cannot skip | Proceed |
| Are accuracy-based metrics excluded? | ✅ Yes — explicitly excluded in section 5.3 | Proceed |

---
## 8. Decisions Log

| # | Decision | Rationale | Alternatives Considered | LEAN Value? |
|---|---|---|---|---|
| 1 | Use AUC-PR as primary metric, not AUC-ROC | AUC-ROC is misleading with 0.17% fraud rate — AUC-PR focuses on minority class | AUC-ROC, F1 at 0.5 | ✅ |
| 2 | Define cost framework in notebook 01, before touching data | Forces business reasoning before modeling — IE framing | Define costs after EDA | ✅ |
| 3 | Exclude SVM and KNN from model comparison | Poor scalability at 284k rows — cost-benefit unfavorable | Include all classifiers | ✅ |
| 4 | Include Isolation Forest as unsupervised benchmark | Demonstrates paradigm judgment — supervised vs unsupervised comparison | Skip unsupervised entirely | ✅ |
| 5 | Parametrize COST_FN and COST_FP as constants | Enables sensitivity analysis and Streamlit slider in nb06 | Hardcode in evaluation | ✅ |

---
## 9. Next Steps — Notebook 02 Preview

**Notebook 02 — Data Understanding** will:

- Load the raw dataset (`creditcard.csv`) from `data/raw/`
- Characterize the class imbalance — visual + quantitative
- Explore PCA features V1–V28 (distributions, outliers, correlation with fraud)
- Analyze `Amount` and `Time` as the only non-anonymized features
- Assess data quality — missing values, duplicates, anomalies
- Produce a data quality report as input to notebook 03

**Lean filter for nb02:** Only explore features that are plausibly predictive of fraud.
No exhaustive EDA of all 30 features — focus on decision-relevant variables.

---

**Next →** [02 — Data Understanding](./02_data_understanding.ipynb)